# DR stage detection - Kaggle runner
Clones the project repo, runs the selected notebooks on GPU and pushes results back to GitHub.
Prereqs: dataset attached, GPU on, internet on, secret `GITHUB_TOKEN` (classic PAT, `repo` scope).
Use **Save Version -> Save & Run All** so it keeps running when the browser tab is closed.

In [ ]:
# 1. Bootstrap: clone repo (token only used for the clone, then removed from git config)
import os, subprocess
from pathlib import Path
from kaggle_secrets import UserSecretsClient

REPO = "https://github.com/pradeesha999/dr-stage-detection-v2.git"
os.environ["GITHUB_TOKEN"] = UserSecretsClient().get_secret("GITHUB_TOKEN")
dst = Path("/kaggle/working/dr_project")
if not dst.exists():
    subprocess.run(["git", "clone", "-q", REPO.replace("https://", f"https://{os.environ['GITHUB_TOKEN']}@"), str(dst)], check=True)
    subprocess.run(["git", "-C", str(dst), "remote", "set-url", "origin", REPO], check=True)
print(subprocess.run(["git", "-C", str(dst), "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

# Trained models (.keras) are git-ignored. When re-running 05 in a new session, add the
# previous run's output as an Input (Add Input -> Your Work) and they are copied in here.
import shutil
(dst / "outputs" / "models").mkdir(parents=True, exist_ok=True)
for f in Path("/kaggle/input").rglob("outputs/models/*.keras"):
    if not (dst / "outputs" / "models" / f.name).exists():
        shutil.copy(f, dst / "outputs" / "models" / f.name); print("copied", f.name)


In [ ]:
# 2. Which notebooks to run (01-03 are fast, 04 is the ~3-4 h training ladder, 05 evaluation)
NOTEBOOKS = "01 02 03 04 05"
!cd /kaggle/working/dr_project && bash scripts/run_notebooks.sh {NOTEBOOKS}

In [ ]:
# 3. Push executed notebooks, figures, metrics back to GitHub
!cd /kaggle/working/dr_project && bash scripts/push_results.sh

In [ ]:
# 4. Keep the trained model as a Kaggle output too (models are git-ignored: too large)
!ls -lh /kaggle/working/dr_project/outputs/models/ /kaggle/working/dr_project/outputs/metrics/ 2>/dev/null | head -40